# Uncertainty evaluation for the measurement of gauge blocks by optical interferometry
---

In [96]:
%load_ext autoreload
%autoreload 2

import sys
import os
# Agrega el directorio padre (la raíz del repositorio) al path temporal de Python
sys.path.append(os.path.abspath(os.path.join('..')))

import numpy as np

from src.refraction_index import EdlenRefractiveIndex

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


La desviación medida entre la longitud nominal y la longitud real del bloque patrón es:

\begin{align}
    d &= l_{real} - L'\\
    d &= l_{fit} -L' + l_t + l_{\omega} + l_A + l_{\Omega} + l_n + l_G + l_{\phi},
\end{align}

Ahora, se definen los parámetros de influencia en la longitud medida y que se reconocen en la ecuación arriba mencionada como sigue:

- $l_{fit}$ es el valor de longitud encontrado por medio del método de fracciones excedentes.
- La corrección por temperatura está dada por la temperatura del bloque patrón ($t_g$), el coeficiente de dilatación termica del material del bloque patrón ($\alpha$). Se expresa como:

\begin{equation}
    l_t = \theta \alpha L,
\end{equation}

donde $\theta$ es el desface de temperatura entendido como $\theta = 20 - t_g$.

In [97]:
# gauge temperature correction
def gauge_temperature_correction(t_g: float, alpha: float, L: float) -> float: return (20.0 - t_g) * alpha * L

- $l_{\omega}$ es la correción atribuida a el espesor de la capa de adherencia. El valor medio de esta correción es cero $<l_{\omega}> = 0 $, debido a que la longitud del bloque contiene este espesor en su definición. Esto por otra parte, no indica que su incertidumbre asociada $u(\omega)$ con la variacion de la capa de adherencia sea cero.
- La correción por errores en el frente de onda resultantes de imperfecciones en las opticas es $l_A$, con una incertumbre asociada $u(l_A)$. El valor esperado de esta correción es cero $<l_A> = 0$, más su incertidumbre es diferente de cero.
- La correción por oblicuidad es una corrección en la longitud que tiene en cuenta el cambio de fase resultante del diseño óptico y alineación inherentes al montaje interferometrico,

\begin{equation}
    l_{\Omega} = \Omega L' = \bigg(\frac{a^2}{16f^2} + \frac{x^2}{2f^2} \bigg)L'
\end{equation}

donde $f$ es la distancia focal del colimador, $a$ es el diametro de la apertura y $x$ es el desplazamiento lateral. 

In [98]:
def obliquity_correction(a: float, f: float, x: float, L: int) -> float: return (a**2/(16 * f**2) + (x**2/(2 * f**2))) * L 

- The refractive index correction is evaluated using a modified version of the Edlén equation:

\begin{equation}
    l_n = (n - 1)L'
\end{equation}

donde $n$ es el índice de refracción del aire evaluado usado una versión de la ecuación de Edlen.

In [99]:
def refractive_index_correction(refractive_index_air: float, L: float) -> float: return (refractive_index_air - 1) * L

- La corrección por geometría del bloque patrón $l_G$ tiene en cuenta la falta de planitud y paralelismo del bloque patrón. La incertudmbre asociadad con esta correción es $u(l_G)$.
- La correción por cambio de fase tiene en cuenta la diferencia aparente entre la longitud de camino óptico y la longitud de camino mecanico. Esta correción compensa la diferencia en la forma en que la luz se refleja en diferentes materiales y superficies. Su incertidumbre es $u(l_{\phi})$. Se calcula como:

\begin{equation}
    l_{\phi} = \frac{1}{n - 1}\bigg(l_{o,p} - \sum_{i=1}^n l_{o,i}\bigg)
\end{equation}

donde $l_{o,p}$ y $l_{o,i}$ son las longitudes ópticas del paquete de bloques y de cada uno de los $i$ bloques individuales. Estas longitudes no deben confundirse con las longitudes mecánicas. Puesto que las últimas ya contienen el termino de correción por cambio de fase. En otras palabras, la longitud óptica es una medida incompleta, es el resultado obtenido por algun método de analisis de franjas (excedentes fraccionarios, phase-stepping, etc.) sin tener en cuenta la correción por cambio de fase. Este valor una vez encontrado se puede agregar a cualquier medición futura de bloques de es mismo material y acabado.

In [100]:
def change_phase_correction(num_blocks: int, optical_length_pack: float, optical_individual_length: tuple[float]) -> float:
    return (1.0 / ( num_blocks - 1.0 )) * (optical_length_pack - np.sum(optical_individual_length))
    

## Wrapp code

In [101]:
import numpy as np
from typing import List

class GaugeBlockLengthCorrection:
    """
    A class to encapsulate the corrections and final true length calculation 
    for a gauge block measured by optical interferometry.
    """
    
    def __init__(self, nominal_length: float, expansion_coefficient: float):
        """
        Initializes the gauge block with its intrinsic properties.
        
        Args:
            nominal_length (float): The nominal length of the gauge block (L').
            expansion_coefficient (float): The thermal expansion coefficient (alpha).
        """
        self.nominal_length = nominal_length
        self.alpha = expansion_coefficient

    def calculate_temperature_correction(self, t_g: float) -> float:
        """
        Calculates the gauge block temperature correction.
        """
        return (20.0 - t_g) * self.alpha * self.nominal_length

    def calculate_obliquity_correction(self, a: float, f: float, x: float) -> float:
        """
        Calculates the obliquity correction due to optical design and alignment.
        """
        return ((a**2) / (16.0 * f**2) + (x**2) / (2.0 * f**2)) * self.nominal_length

    def calculate_refractive_index_correction(self, refractive_index: float) -> float:
        """
        Calculates the refractive index correction using the evaluated index of air.
        """
        return (refractive_index - 1.0) * self.nominal_length

    @staticmethod
    def calculate_phase_change_correction(num_blocks: int, optical_length_pack: float, optical_individual_lengths: List[float]) -> float:
        """
        Calculates the phase change correction using the pack experiment method.
        Note: This is a static method as it depends on a batch of blocks rather 
        than a single block's intrinsic properties.
        """
        sum_individual_lengths = np.sum(optical_individual_lengths)
        return (1.0 / (num_blocks - 1)) * (optical_length_pack - sum_individual_lengths)

    def calculate_true_length(self, l_fit: float, t_g: float, a: float, f: float, x: float, 
                              refractive_index: float, l_w: float, l_A: float, l_G: float, l_phi: float) -> float:
        """
        Computes the final true length of the gauge block by applying all corrections.
        """
        # Calculate dynamic corrections
        l_t = self.calculate_temperature_correction(t_g)
        l_Omega = self.calculate_obliquity_correction(a, f, x)
        l_n = self.calculate_refractive_index_correction(refractive_index)
        
        # Total deviation calculation: d = l_fit - L' + l_t + l_w + l_A + l_Omega + l_n + l_G + l_phi
        # Since l_real = L' + d, we can simplify to:
        # l_real = l_fit + l_t + l_w + l_A + l_Omega + l_n + l_G + l_phi
        
        l_real = l_fit + l_t + l_w + l_A + l_Omega + l_n + l_G + l_phi
        
        return {
            "l_fit": l_fit,
            "l_t": l_t,
            "l_Omega": l_Omega,
            "l_n": l_n,
            "l_phi": l_phi,
            "l_real": l_real
        }

In [102]:
gauge_block_100mm = GaugeBlockLengthCorrection(nominal_length=100.0, expansion_coefficient=11.5e-6)

l_phi = GaugeBlockLengthCorrection.calculate_phase_change_correction(
        num_blocks=4, 
        optical_length_pack=400.0001, 
        optical_individual_lengths=[100.00002, 100.00001, 100.00003, 100.00001]
)

true_length = gauge_block_100mm.calculate_true_length(
    l_fit=100.00005,
    t_g=20.1,
    a=0.5,
    f=500.0,
    x=0.00,        #  x = 0, characteristic of the Twyman Green interferometer
    refractive_index=1.00026,
    l_w=0.0,       # Adherence layer 
    l_A=0.0,       # Wavefront error expected value
    l_G=0.0,       # Geometry correction
    l_phi=l_phi
)

print(f"True length of the gauge block: {true_length["l_real"]:.6f} mm")

True length of the gauge block: 100.025951 mm


In [103]:
l_phi * 1e6

np.float64(9.999999974752427)

In [104]:
(true_length["l_real"] - 100.0) * 1e3

np.float64(25.95124999997722)

---
### Ejemplo 2

In [105]:
# Parámetros de la Medición Óptica y Ambientales
# Radio de apertura (a): Tomamos el MFD interpolado para 543 nm (~3.5 um), el radio es ~1.75 um = 0.00175 mm
a_apertura = 0.00175 # mm
f_lente = 500.0 # mm
x_desplazamiento = 0.0 # Configuración Twyman-Green asume eje óptico perfecto
t_g = 20.0 # °C
n_aire = 1.000200738

In [106]:
# Definición de Parámetros del Bloque Principal (100 mm, Acero)
L_nominal = 100.0                       # [mm]
l_fit_medido = 100.0000299499           # [mm] Obtenido por EFSW script propio
alpha_acero = 11.5e-6                   # [K^-1] (Coeficiente típico para bloques de acero)
bloque_100mm = GaugeBlockLengthCorrection(nominal_length=L_nominal, expansion_coefficient=alpha_acero)

# Datos para el Experimento de Paquete (Cambio de Fase)
# Bloques: 2mm, 5mm, 8mm, 10mm (n = 4)
num_blocks = 4
# Longitudes ópticas individuales (simuladas con un ligero desvío por rugosidad/fase)
l_opt_ind = [2.000015, 5.000010, 8.000012, 10.000018] 
suma_l_opt_ind = sum(l_opt_ind) # 25.000055 mm

# Forzamos que la diferencia nos de un l_phi de -0.000025 mm (-25 nm)
# l_phi = (l_opt_pack - suma_l_opt_ind) / 3  => l_opt_pack = suma_l_opt_ind + (3 * -0.000025)
l_opt_pack = 24.999980 # mm

In [107]:
l_phi_calculado = GaugeBlockLengthCorrection.calculate_phase_change_correction(
    num_blocks=num_blocks, 
    optical_length_pack=l_opt_pack, 
    optical_individual_lengths=l_opt_ind
)
print(l_phi_calculado * 1e6)

-24.999999999645674


In [108]:
resultados = bloque_100mm.calculate_true_length(
    l_fit=l_fit_medido,
    t_g=t_g,
    a=a_apertura,
    f=f_lente,
    x=x_desplazamiento,
    refractive_index=n_aire,
    l_w=0.0,  # Capa de adherencia asumida en 0
    l_A=0.0,  # Error de frente de onda asumido en 0
    l_G=0.0,  # Corrección por geometría asumida en 0
    l_phi=l_phi_calculado
)

print("--- REPORTE DE CALIBRACIÓN DE BLOQUE PATRÓN ---")
print(f"Longitud Nominal: {L_nominal} mm")
print(f"Valor inicial medido (l_fit): {resultados['l_fit']:.8f} mm\n")
print("CORRECCIONES APLICADAS:")
print(f"  Cambio de Fase (l_phi): {resultados['l_phi']:.8f} mm ({-resultados['l_phi']*1e6:.1f} nm)")
print(f"  Temperatura (l_t):      {resultados['l_t']:.8f} mm (A 20.0 °C la corrección es nula)")
print(f"  Oblicuidad (l_Omega):   {resultados['l_Omega']:.12f} mm (Efecto de la lente y fibra)")
print(f"  Índice Refracción (l_n): {resultados['l_n']:.8f} mm\n")
print(f"LONGITUD REAL FINAL:      {resultados['l_real']:.8f} mm")

--- REPORTE DE CALIBRACIÓN DE BLOQUE PATRÓN ---
Longitud Nominal: 100.0 mm
Valor inicial medido (l_fit): 100.00002995 mm

CORRECCIONES APLICADAS:
  Cambio de Fase (l_phi): -0.00002500 mm (25.0 nm)
  Temperatura (l_t):      0.00000000 mm (A 20.0 °C la corrección es nula)
  Oblicuidad (l_Omega):   0.000000000077 mm (Efecto de la lente y fibra)
  Índice Refracción (l_n): 0.02007380 mm

LONGITUD REAL FINAL:      100.02007875 mm


In [109]:
(resultados['l_fit'] - L_nominal)*1e6

29.949899996495333

In [110]:
(resultados['l_real'] - L_nominal)*1e6

np.float64(20078.74997656245)